# Lab Manual 1 — Dataset Inspection & Data Profiling
**Course:** Data Analysis and Visualization
**Dataset:** UCI Adult Income (`adult.csv`)

This notebook covers initial dataset inspection, data-quality checks, cleaning decisions, descriptive/frequency analysis, SQL-based extraction, and a final profiling summary.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

## Loading & Initial Setup

In [7]:

df = pd.read_csv('adult.csv')
df.head()


HI


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 6.2 MB


In [6]:
df.shape

(32561, 15)

## Task 1 — Fix Hidden Missing Values



In [8]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [9]:
# Confirm the '?' placeholder is present
df['workclass'].unique()

<ArrowStringArray>
[               '?',          'Private',        'State-gov',
      'Federal-gov', 'Self-emp-not-inc',     'Self-emp-inc',
        'Local-gov',      'Without-pay',     'Never-worked']
Length: 9, dtype: str

In [10]:
# Replace all '?' placeholders with proper NaN
df.replace('?', np.nan, inplace=True)

# Re-check missing values now that they are real NaNs
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

## Task 2 — Quantify Missingness

Count and percentage of missing values per column, sorted descending.

In [11]:
missing_count = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0]
missing_summary

,missing_count,missing_pct
occupation,1843,5.660146
workclass,1836,5.638647
native.country,583,1.790486


**Observation:** the columns with the most missing data are `occupation`, `workclass`,
and `native.country` — consistent with the columns flagged in the lab manual.

## Task 3 — Handle Missing Values

- **`workclass`** and **`occupation`** (categorical): imputed with the string `"Unknown"` rather
  than dropped. Dropping ~1,800+ rows for each column would discard a meaningful chunk of the
  dataset (~5-6%), and "Unknown" preserves that these people simply didn't report a
  workclass/occupation (a large share is not-in-labor-force / never-worked), which is itself
  informative rather than noise.
- **`native.country`**: also categorical with missing values; imputed with the **mode**
  (`United-States`) since the column is heavily dominated by one category (~90%+), so the mode
  is a reasonable, low-bias fill.
- **Numeric columns**: none of the numeric columns (`age`, `fnlwgt`, `education.num`,
  `capital.gain`, `capital.loss`, `hours.per.week`) have missing values in this dataset, so no
  numeric imputation is needed.

In [12]:
df['workclass'] = df['workclass'].fillna('Unknown')
df['occupation'] = df['occupation'].fillna('Unknown')
df['native.country'] = df['native.country'].fillna(df['native.country'].mode()[0])

# Confirm no missing values remain
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

## Task 4 — Detect Duplicates

In [13]:
# Exact duplicate rows
exact_dupes = df.duplicated().sum()
print(f'Exact duplicate rows: {exact_dupes}')

Exact duplicate rows: 24


In [14]:
# Duplicates ignoring the target/label column ('income')
cols_no_label = [c for c in df.columns if c != 'income']
dupes_no_label = df.duplicated(subset=cols_no_label).sum()
print(f'Duplicate rows ignoring income label: {dupes_no_label}')

Duplicate rows ignoring income label: 25


In [15]:
df = df.drop_duplicates()
df.shape

(32537, 15)

## Task 5 — Fix Inconsistent Categorical Data

In [16]:
for col in ['education', 'marital.status', 'native.country']:
    print(col, '->', df[col].unique())
    print()

education -> <ArrowStringArray>
[     'HS-grad', 'Some-college',      '7th-8th',         '10th',
    'Doctorate',  'Prof-school',    'Bachelors',      'Masters',
         '11th',   'Assoc-acdm',    'Assoc-voc',      '1st-4th',
      '5th-6th',         '12th',          '9th',    'Preschool']
Length: 16, dtype: str

marital.status -> <ArrowStringArray>
[              'Widowed',              'Divorced',             'Separated',
         'Never-married',    'Married-civ-spouse', 'Married-spouse-absent',
     'Married-AF-spouse']
Length: 7, dtype: str

native.country -> <ArrowStringArray>
[             'United-States',                     'Mexico',
                     'Greece',                    'Vietnam',
                      'China',                     'Taiwan',
                      'India',                'Philippines',
            'Trinadad&Tobago',                     'Canada',
                      'South',         'Holand-Netherlands',
                'Puerto-Rico',             

In [17]:
# Strip leading/trailing whitespace from all object (string) columns
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    df[col] = df[col].str.strip()

# Re-check after stripping
for col in ['education', 'marital.status', 'native.country']:
    print(col, '->', df[col].unique())

education -> <ArrowStringArray>
[     'HS-grad', 'Some-college',      '7th-8th',         '10th',
    'Doctorate',  'Prof-school',    'Bachelors',      'Masters',
         '11th',   'Assoc-acdm',    'Assoc-voc',      '1st-4th',
      '5th-6th',         '12th',          '9th',    'Preschool']
Length: 16, dtype: str
marital.status -> <ArrowStringArray>
[              'Widowed',              'Divorced',             'Separated',
         'Never-married',    'Married-civ-spouse', 'Married-spouse-absent',
     'Married-AF-spouse']
Length: 7, dtype: str
native.country -> <ArrowStringArray>
[             'United-States',                     'Mexico',
                     'Greece',                    'Vietnam',
                      'China',                     'Taiwan',
                      'India',                'Philippines',
            'Trinadad&Tobago',                     'Canada',
                      'South',         'Holand-Netherlands',
                'Puerto-Rico',               

C:\Users\Zeyan\AppData\Local\Temp\ipykernel_9988\479013174.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns


**Observation:** in this particular CSV export the categorical values did not have visible
leading/trailing whitespace (the values already looked clean), but `.str.strip()` was still
applied defensively to every text column since inconsistent whitespace is a known issue with
this dataset in other distributions (e.g. the raw UCI `.data` file).

## Task 6 — Descriptive Statistics

In [15]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32537.000000,3.253700e+04,32537.000000,32537.000000,32537.000000,32537.000000
mean,38.585549,1.897808e+05,10.081815,1078.443741,87.368227,40.440329
std,13.637984,1.055565e+05,2.571633,7387.957424,403.101833,12.346889
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.369930e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [19]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32537.000000,3.253700e+04,32537.000000,32537.000000,32537.000000,32537.000000
mean,38.585549,1.897808e+05,10.081815,1078.443741,87.368227,40.440329
std,13.637984,1.055565e+05,2.571633,7387.957424,403.101833,12.346889
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.369930e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [20]:
mean_age = df['age'].mean()
median_age = df['age'].median()
mean_hours = df['hours.per.week'].mean()
median_hours = df['hours.per.week'].median()

print(f'Age      -> mean: {mean_age:.2f}, median: {median_age}')
print(f'Hours/wk -> mean: {mean_hours:.2f}, median: {median_hours}')

Age      -> mean: 38.59, median: 37.0
Hours/wk -> mean: 40.44, median: 40.0


## Task 7 — Frequency Analysis

In [21]:
# (a) Most common occupation
occupation_counts = df['occupation'].value_counts()
print('Most common occupation:', occupation_counts.idxmax(), f'({occupation_counts.max()} records)')
occupation_counts

Most common occupation: Prof-specialty (4136 records)


occupation
Prof-specialty       4136
Craft-repair         4094
Exec-managerial      4065
Adm-clerical         3768
Sales                3650
Other-service        3291
Machine-op-inspct    2000
Unknown              1843
Transport-moving     1597
Handlers-cleaners    1369
Farming-fishing       992
Tech-support          927
Protective-serv       649
Priv-house-serv       147
Armed-Forces            9
Name: count, dtype: int64

In [19]:
# (b) Percentage distribution of sex
df['sex'].value_counts(normalize=True) * 100

sex
Male      66.92381
Female    33.07619
Name: proportion, dtype: float64

In [20]:
# (c) Percentage distribution of income label
df['income'].value_counts(normalize=True) * 100

income
<=50K    75.907428
>50K     24.092572
Name: proportion, dtype: float64

## Task 8 — Cross-Check a Data Quality Assumption

Verify that `education` and `education.num` are consistent — i.e., each education level always
maps to the same numeric code.

In [22]:
edu_check = df.groupby('education')['education.num'].unique()
edu_check

education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education.num, dtype: object

In [23]:
inconsistent = edu_check[edu_check.apply(len) > 1]
if inconsistent.empty:
    print('No inconsistencies found: every education level maps to exactly one education.num value.')
else:
    print('Inconsistent mappings found:')
    print(inconsistent)

No inconsistencies found: every education level maps to exactly one education.num value.


## Task 9 — SQL-Based Data Extraction

Load the cleaned CSV data into a local SQLite database table, then extract data using a SQL
query and load the result back into a pandas DataFrame via `pd.read_sql_query`.

In [24]:
# Create an in-memory (or file-based) SQLite database and load the DataFrame into it
conn = sqlite3.connect('adult_income.db')
df.to_sql('adult_income', conn, if_exists='replace', index=False)

# Example query: extract records where age > 30
query = 'SELECT * FROM adult_income WHERE age > 30;'
df_sql = pd.read_sql_query(query, conn)

df_sql.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,Unknown,77053,HS-grad,9,Widowed,Unknown,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,Unknown,186061,Some-college,10,Widowed,Unknown,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [25]:
print('Original cleaned DataFrame shape:', df.shape)
print('SQL query result shape (age > 30):     ', df_sql.shape)
print('Rows with age > 30 in original df:     ', (df['age'] > 30).sum())

assert df_sql.shape[0] == (df['age'] > 30).sum(), 'Row count mismatch between SQL result and pandas filter!'
print('\nSQL extraction confirmed correct: row counts match.')

Original cleaned DataFrame shape: (32537, 15)
SQL query result shape (age > 30):      (21979, 15)
Rows with age > 30 in original df:      21979

SQL extraction confirmed correct: row counts match.


In [26]:
conn.close()

## Task 10 — Data Profiling Summary Report

**Dataset overview**
The dataset is the UCI Adult Income dataset, containing 32,561 rows and 15 columns after
loading, representing individual demographic and employment records (age, workclass,
education, occupation, marital status, race, sex, capital gain/loss, hours worked per week,
native country) along with a binary income label (`<=50K` / `>50K`).

**Data quality issues found**
- **Hidden missing values:** missing data was encoded as the string `"?"` rather than `NaN`,
  so `isnull().sum()` initially reported zero missing values. After replacement, three columns
  had real missing data: `occupation` (1,843), `workclass` (1,836), and `native.country` (583).
- **Duplicates:** 24 exact duplicate rows (identical across all columns including `income`)
  were found and removed. An additional 1 row matched on all columns except `income`, but was
  kept since differing income for otherwise-identical attributes is plausible, not redundant.
- **Inconsistent categorical data:** categorical columns were checked for inconsistent casing/
  whitespace via `.unique()`; whitespace stripping was applied defensively across all text
  columns as a precaution, since this is a documented issue in other versions of this dataset.

**Key observations**
- The income classes are imbalanced (~3:1 in favor of `<=50K`), which should inform any later
  modeling choices (metric selection, resampling, class weighting).
- `education` and `education.num` are perfectly consistent — every education level maps to
  exactly one numeric code, so either column can be used interchangeably for ordinal encoding.
- Mean and median for `hours.per.week` are nearly identical (~40), showing most people report
  a standard full-time week, while `age` shows a mild right skew.

**Cleaning decisions made**
- Replaced `"?"` placeholders with `NaN` across the dataset.
- Imputed `workclass` and `occupation` missing values with `"Unknown"` (informative category
  rather than a discard).
- Imputed `native.country` missing values with the mode (`United-States`), given its heavy
  class dominance.
- Dropped 24 exact duplicate rows; kept near-duplicates that differed only in `income`.
- Stripped whitespace from all text columns as a defensive cleaning step.

## Save Cleaned Dataset

In [27]:
df.to_csv('adult_cleaned.csv', index=False)
print('Cleaned dataset saved as adult_cleaned.csv')
print('Final shape:', df.shape)

Cleaned dataset saved as adult_cleaned.csv
Final shape: (32537, 15)
